Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 07 · Join Drishti's data with your own in DuckDB

Arrow tables are DuckDB's native input: read from Flight, then join with data of your own (here two small, made-up tables) in SQL on your machine. Your data never goes to the server; the server's data comes to you as your person may see it.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## Drishti's side: trades and a P&L vector

In [ ]:
f = connect()
trades = f.kind("trade", paths=["TRADE_ID", "TT_GROUP", "CURRENCY", "QUANTITY"], date=DATE)
pnl = f.measure("var.scenarios", SCOPE, date=DATE)
print(trades.num_rows, "trades;", pnl.num_rows, "scenarios")

## Your side

Two tables of your own, made up for the example: rates to USD you keep for a report, and a benchmark's daily P&L on the same historical dates (synthetic, seeded).

In [ ]:
import numpy as np
import pandas as pd

my_rates = pd.DataFrame({"CURRENCY": ["USD", "EUR", "GBP", "JPY", "CAD", "CHF", "AUD"],
                         "TO_USD":   [1.0, 1.08, 1.27, 0.0066, 0.73, 1.13, 0.66]})
rng = np.random.default_rng(7)
dates = pnl.column("SCENARIO_DATE").to_pylist()
my_benchmark = pd.DataFrame({"SCENARIO_DATE": dates, "BENCH_PNL": rng.normal(0, 1e6, len(dates))})

## Join in SQL

DuckDB reads the Arrow table and the pandas frames by their variable names. A currency missing from your rates stays visible (a left join) rather than dropping trades silently.

In [ ]:
import duckdb

duckdb.sql("""
    SELECT t.TT_GROUP                         AS desk,
           count(*)                           AS trades,
           round(sum(t.QUANTITY * r.TO_USD) / 1e9, 2) AS notional_usd_bn,
           count(*) FILTER (WHERE r.TO_USD IS NULL)   AS trades_without_rate
    FROM trades t LEFT JOIN my_rates r USING (CURRENCY)
    GROUP BY 1 ORDER BY notional_usd_bn DESC NULLS LAST
""").df()

## Your benchmark against the desk's P&L

The scenario dates are the join key: the desk's P&L on each historical date beside your benchmark's, and their correlation.

In [ ]:
duckdb.sql("""
    SELECT corr(p.PNL, b.BENCH_PNL)            AS correlation,
           count(*)                            AS scenarios,
           min(p.PNL)                          AS worst_desk_pnl
    FROM pnl p JOIN my_benchmark b USING (SCENARIO_DATE)
""").df()

In [ ]:
f.close()